In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
floc1='/content/drive/MyDrive/DM_2026/'

In [ ]:
!pip install -q pandas numpy torch transformers gensim tqdm

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import numpy as np

# Optimized to accept pre-loaded models and tokenizers
def get_embeddings(df, column_name, codebert_tokenizer, codebert_model, bert_tokenizer, bert_model, batch_size=16):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    texts = df[column_name].astype(str).tolist()

    codebert_embeddings = []
    bert_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc=f"Generating {column_name} Embeddings"):
        batch_texts = texts[i:i+batch_size]

        # CodeBERT Embeddings
        cb_inputs = codebert_tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        with torch.no_grad():
            cb_outputs = codebert_model(**cb_inputs)
            cb_batch_emb = cb_outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            codebert_embeddings.append(cb_batch_emb)

        # BERT Embeddings
        b_inputs = bert_tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        with torch.no_grad():
            b_outputs = bert_model(**b_inputs)
            b_batch_emb = b_outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            bert_embeddings.append(b_batch_emb)

    return np.vstack(codebert_embeddings), np.vstack(bert_embeddings)

In [ ]:
# Overwriting with correct and robust embedding extraction for BERT and CodeBERT
def get_embeddings(df, column_name, codebert_tokenizer, codebert_model, bert_tokenizer, bert_model, batch_size=16):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    texts = df[column_name].astype(str).tolist()

    codebert_embeddings = []
    bert_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc=f"Generating {column_name} Embeddings"):
        batch_texts = texts[i:i+batch_size]

        # CodeBERT Embeddings (Using the CLS token which represents the sequence summary)
        cb_inputs = codebert_tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        with torch.no_grad():
            cb_outputs = codebert_model(**cb_inputs)
            # Use [CLS] token (first token of the sequence) for CodeBERT classification/representation task
            cb_batch_emb = cb_outputs.last_hidden_state[:, 0, :].cpu().numpy()
            codebert_embeddings.append(cb_batch_emb)

        # BERT Embeddings (Using CLS token representation)
        b_inputs = bert_tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        with torch.no_grad():
            b_outputs = bert_model(**b_inputs)
            b_batch_emb = b_outputs.last_hidden_state[:, 0, :].cpu().numpy()
            bert_embeddings.append(b_batch_emb)

    return np.vstack(codebert_embeddings), np.vstack(bert_embeddings)

In [ ]:

import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec
import gensim.downloader as api
def generate_embeddings(
    df,
    column_name,
    max_features=500,
    vector_size=200,
    window=5,
    min_count=1,
    workers=4
):
    """
    Generate text embeddings using:
    1. CountVectorizer
    2. TF-IDF
    3. GloVe Wikipedia
    4. GloVe Twitter
    5. Word2Vec trained on the input dataset

    Returns a dictionary of embedding lists.
    """

    if column_name not in df.columns:
        raise ValueError(f"Column '{column_name}' not found in DataFrame.")

    # Handle missing values and normalize input
    texts = df[column_name].fillna("").astype(str).tolist()

    # Basic tokenization for word embeddings
    tokenized = [
        re.findall(r"\b\w+\b", text.lower())
        for text in texts
    ]

    results = {}

    # 1. Count Vectorizer
    count_vectorizer = CountVectorizer(
        max_features=500,ngram_range=(1,3)
    )

    try:
        results["count_vector"] = (
            count_vectorizer.fit_transform(texts)
            .toarray()
            .tolist()
        )
    except ValueError:
        results["count_vector"] = [[] for _ in texts]

    # 2. TF-IDF Vectorizer
    tfidf_vectorizer = TfidfVectorizer(
        max_features=500, ngram_range=(1,3)
    )

    try:
        results["tfidf_vector"] = (
            tfidf_vectorizer.fit_transform(texts)
            .toarray()
            .tolist()
        )
    except ValueError:
        results["tfidf_vector"] = [[] for _ in texts]

    # 3 and 4. Pretrained GloVe embeddings
    glove_models = {
        "glove_wiki": "glove-wiki-gigaword-200",
        "glove_twitter": "glove-twitter-200"
    }

    def average_pretrained_vectors(tokens, model):
        vectors = [
            model[word]
            for word in tokens
            if word in model
        ]

        if not vectors:
            return np.zeros(model.vector_size).tolist()

        return np.mean(vectors, axis=0).tolist()

    for name, model_name in glove_models.items():
        try:
            embedding_model = api.load(model_name)

            results[name] = [
                average_pretrained_vectors(tokens, embedding_model)
                for tokens in tokenized
            ]

        except Exception as e:
            print(f"{name} failed: {e}")
            results[name] = None

    # 5. Train Word2Vec on the supplied dataset
    try:
        nonempty_tokens = [tokens for tokens in tokenized if tokens]

        if nonempty_tokens:
            w2v_model = Word2Vec(
                sentences=nonempty_tokens,
                vector_size=vector_size,
                window=window,
                min_count=min_count,
                workers=workers,
                seed=42
            )

            results["word2vec"] = [
                average_pretrained_vectors(tokens, w2v_model.wv)
                for tokens in tokenized
            ]
        else:
            results["word2vec"] = [
                np.zeros(vector_size).tolist()
                for _ in texts
            ]

    except Exception as e:
        print(f"Word2Vec failed: {e}")
        results["word2vec"] = None

    return results

In [ ]:
inp=input("enter your project id")
inp=int(inp)
embv=[]
astm=["pre","post","ino","dfs","bfs"]
embn=["count_vector","tfidf_vector","glove_wiki","glove_twitter","word2vec"]
fname=floc1+str(inp)+'_asgnstud.xlsx'
df=pd.read_excel(fname)
embv.append(df.iloc[:,:60].values)
embv.append(df.iloc[:,60:80].values)

# Load tokenizers and models once here to save time and memory
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Loading models on {device}...")
codebert_tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
codebert_model = AutoModel.from_pretrained("microsoft/codebert-base").to(device)
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModel.from_pretrained("bert-base-uncased").to(device)

for i in range(0,5):
  results=generate_embeddings(df,astm[i])
  a,b=get_embeddings(df,astm[i], codebert_tokenizer, codebert_model, bert_tokenizer, bert_model)
  for j in range(0,5):
    embv.append(results[embn[j]])
  embv.append(a)
  embv.append(b)

Loading models on cpu...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[==================================================] 100.0% 252.1/252.1MB downloaded
[==================================================] 100.0% 758.5/758.5MB downloaded


Generating bfs Embeddings: 100%|██████████| 9/9 [09:30<00:00, 63.44s/it]


In [ ]:
for i in range(0,len(embv)):
  fname=floc1+str(37*(inp-1)+i+1)+'_vec.csv'
  np.savetxt(fname,embv[i],delimiter=',',fmt='%f')

In [ ]:
for i in range(0,len(embv)):
  fname=floc1+str(i)+'_vec.csv'
  data=np.genfromtxt(fname,delimiter=',')
  print(data.shape,i)

(130, 60) 0
(130, 60) 1
(130, 20) 2
(130, 500) 3
(130, 500) 4
(130, 200) 5
(130, 200) 6
(130, 200) 7
(130, 768) 8
(130, 768) 9
(130, 500) 10
(130, 500) 11
(130, 200) 12
(130, 200) 13
(130, 200) 14
(130, 768) 15
(130, 768) 16
(130, 500) 17
(130, 500) 18
(130, 200) 19
(130, 200) 20
(130, 200) 21
(130, 768) 22
(130, 768) 23
(130, 500) 24
(130, 500) 25
(130, 200) 26
(130, 200) 27
(130, 200) 28
(130, 768) 29
(130, 768) 30
(130, 500) 31
(130, 500) 32
(130, 200) 33
(130, 200) 34
(130, 200) 35
(130, 768) 36


In [ ]:
for i in range(0,118):
  for j in range(0,37):
    fname=floc1+str(37*i+j+1)+'_vec.csv'
    try:
      data=np.genfromtxt(fname,delimiter=',')
    except:
      print(i+1)
      break

2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
